<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/DL-2026/Lecture_2/Lecture_2_1_Advanced_Topics_in_Multilayer_Perceptron_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Часть II. Дополнительные вопросы обучения многослойного перцептрона

### 1. Введение

В первой части были рассмотрены базовая архитектура MLP, прямой проход и алгоритм обратного распространения ошибки для трёх конкретных примеров, а также введено понятие локального градиента $\delta$. Настоящая, вторая часть посвящена более широкому кругу вопросов, возникающих при практическом построении и обучении многослойных перцептронов. Мы рассмотрим различные функции активации, функции потерь, методы регуляризации, способы инициализации весов, продвинутые алгоритмы оптимизации, проблему исчезающего и взрывающегося градиента, обучение по мини-батчам, особенности многоклассовой классификации, практические рекомендации и теорему универсальной аппроксимации. Изложение сохраняет научный стиль первой части и предполагает знакомство с основными понятиями MLP и backpropagation.

---

### 2. Функции активации

Функция активации вносит нелинейность в модель и определяет выход нейрона по его линейной комбинации. Выбор активации существенно влияет на обучаемость сети и качество решения задачи.

#### 2.1. Сигмоидная функция

Сигмоидная функция уже использовалась в первой части:
$$
\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad \sigma'(z) = \sigma(z)\bigl(1 - \sigma(z)\bigr).
$$
Область значений $(0,1)$ делает её подходящей для выходного слоя в задачах бинарной классификации (в паре с кросс-энтропийной функцией потерь). Однако для скрытых слоёв она имеет два серьёзных недостатка. Во-первых, производная $\sigma'(z)$ достигает максимума $0.25$ при $z=0$ и быстро стремится к нулю при удалении от нуля, что приводит к **исчезающему градиенту** при глубоких сетях. Во-вторых, выходы сигмоиды не центрированы относительно нуля (среднее положительно), что может замедлять сходимость градиентного спуска.

#### 2.2. Гиперболический тангенс (tanh)

Функция гиперболического тангенса определяется как
$$
\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}} = 2\sigma(2z) - 1,
$$
и её производная
$$
\tanh'(z) = 1 - \tanh^2(z).
$$
Область значений $(-1,1)$ делает выходы центрированными относительно нуля, что обычно ускоряет сходимость по сравнению с сигмоидой. Тем не менее проблема исчезающего градиента сохраняется, так как производная также стремится к нулю при больших по модулю значениях $z$.

#### 2.3. ReLU (Rectified Linear Unit)

Функция ReLU задаётся выражением
$$
\mathrm{ReLU}(z) = \max(0, z),
$$
с производной
$$
\mathrm{ReLU}'(z) =
\begin{cases}
1, & z > 0,\\
0, & z \leq 0.
\end{cases}
$$
ReLU значительно ускоряет обучение глубоких сетей благодаря тому, что для положительных $z$ градиент не затухает (производная равна 1). Однако при $z \leq 0$ градиент равен нулю, что приводит к «мёртвым нейронам», которые никогда не активируются. Для смягчения этой проблемы используются варианты ReLU.

#### 2.4. Leaky ReLU и Parametric ReLU

Leaky ReLU вводит небольшой наклон для отрицательной области:
$$
\mathrm{LeakyReLU}(z) =
\begin{cases}
z, & z > 0,\\
\alpha z, & z \leq 0,
\end{cases}
\qquad
\mathrm{LeakyReLU}'(z) =
\begin{cases}
1, & z > 0,\\
\alpha, & z \leq 0,
\end{cases}
$$
где $\alpha$ — малая константа (например, $0.01$). В Parametric ReLU параметр $\alpha$ обучается вместе с другими параметрами сети. Эти функции позволяют избежать полной «смерти» нейронов и сохраняют градиент для отрицательных входов.

#### 2.5. Softmax

Для задач многоклассовой классификации на выходном слое часто используется функция softmax, преобразующая вектор логитов $\mathbf z = (z_1, \dots, z_C)^\top$ в распределение вероятностей:
$$
\mathrm{softmax}(\mathbf z)_i = \frac{e^{z_i}}{\sum_{k=1}^{C} e^{z_k}}, \quad i = 1, \dots, C.
$$
Производная softmax имеет матричный вид. Если обозначить $\mathbf s = \mathrm{softmax}(\mathbf z)$, то
$$
\frac{\partial s_i}{\partial z_j} = s_i (\delta_{ij} - s_j),
$$
где $\delta_{ij}$ — символ Кронекера. Это выражение играет важную роль при выводе градиентов для кросс-энтропийной функции потерь (см. раздел 9).

---

### 3. Функции потерь

Функция потерь количественно оценивает расхождение между предсказаниями сети и истинными метками. В первой части использовалась квадратичная ошибка (MSE), однако для классификации более естественны кросс-энтропийные функции.

#### 3.1. Среднеквадратичная ошибка (MSE)

Для задачи регрессии с целевым вектором $\mathbf y$ и предсказанием $\hat{\mathbf y}$:
$$
L_{\text{MSE}} = \frac{1}{2} \|\mathbf y - \hat{\mathbf y}\|^2.
$$
Её градиент по выходу равен
$$
\frac{\partial L_{\text{MSE}}}{\partial \hat{\mathbf y}} = \hat{\mathbf y} - \mathbf y.
$$
MSE чувствительна к выбросам и может приводить к медленной сходимости при использовании сигмоидной активации на выходе из-за насыщения.

#### 3.2. Бинарная кросс-энтропия

Для бинарной классификации ($C=2$) с истинной меткой $y \in \{0,1\}$ и предсказанной вероятностью $\hat y \in (0,1)$:
$$
L_{\text{BCE}} = -\bigl[ y \log \hat y + (1-y) \log (1-\hat y) \bigr].
$$
Градиент по логиту $z$ (до сигмоиды) имеет простую форму. Если $\hat y = \sigma(z)$, то
$$
\frac{\partial L_{\text{BCE}}}{\partial z} = \hat y - y.
$$
Это свойство делает пару «сигмоида + бинарная кросс-энтропия» особенно удобной: градиент по логиту не содержит множителя $\sigma'(z)$ и не затухает при насыщении.

#### 3.3. Категориальная кросс-энтропия

Для многоклассовой классификации с $C$ классами истинное распределение обычно задаётся one-hot вектором $\mathbf y = (y_1, \dots, y_C)^\top$, где $y_i \in \{0,1\}$ и $\sum_i y_i = 1$. Предсказание $\hat{\mathbf y} = \mathrm{softmax}(\mathbf z)$. Категориальная кросс-энтропия определяется как
$$
L_{\text{CCE}} = - \sum_{i=1}^{C} y_i \log \hat y_i.
$$
Градиент по логитам $\mathbf z$ равен
$$
\frac{\partial L_{\text{CCE}}}{\partial \mathbf z} = \hat{\mathbf y} - \mathbf y,
$$
что также имеет элегантный вид и широко используется на практике.

---

### 4. Регуляризация

Регуляризация применяется для предотвращения переобучения и улучшения обобщающей способности модели.

#### 4.1. L2-регуляризация (weight decay)

К функции потерь добавляется штраф за большие веса:
$$
\tilde L = L + \frac{\lambda}{2} \sum_{\ell} \|\mathbf W^{(\ell)}\|_F^2,
$$
где $\|\cdot\|_F$ — норма Фробениуса, $\lambda > 0$ — коэффициент регуляризации. Тогда градиент по весам слоя $\ell$ принимает вид
$$
\frac{\partial \tilde L}{\partial \mathbf W^{(\ell)}} = \frac{\partial L}{\partial \mathbf W^{(\ell)}} + \lambda \mathbf W^{(\ell)}.
$$
Это приводит к экспоненциальному затуханию весов, поэтому метод называют weight decay.

#### 4.2. L1-регуляризация

В случае L1-регуляризации штраф пропорционален сумме модулей весов:
$$
\tilde L = L + \lambda \sum_{\ell} \|\mathbf W^{(\ell)}\|_1,
$$
где $\|\mathbf W^{(\ell)}\|_1 = \sum_{i,j} |w_{ij}^{(\ell)}|$. Градиент по элементам весов:
$$
\frac{\partial \tilde L}{\partial w_{ij}^{(\ell)}} = \frac{\partial L}{\partial w_{ij}^{(\ell)}} + \lambda \, \mathrm{sign}(w_{ij}^{(\ell)}).
$$
L1-регуляризация способствует разреживанию весов, обращая многие из них в ноль, что может использоваться для отбора признаков.

#### 4.3. Dropout

**Идея.** Во время обучения каждый нейрон скрытого слоя с вероятностью $p$ «отключается» (его выход обнуляется). Это предотвращает коадаптацию нейронов и действует как усреднение множества подсетей.

**Прямой проход с Dropout.**  
Пусть $\mathbf h$ — выход слоя (после активации). Генерируется маска $\mathbf d$ из независимых бернуллиевских случайных величин:
$$
d_j =
\begin{cases}
0, & \text{с вероятностью } p,\\
1, & \text{с вероятностью } 1-p.
\end{cases}
$$
Выход слоя с dropout:
$$
\mathbf h' = \mathbf d \odot \mathbf h,
$$
где $\odot$ — поэлементное умножение. Для батча из $m$ примеров формируется матрица масок $\mathbf D$ размера $m \times k$ ($k$ — число нейронов), и выход:
$$
\mathbf H' = \mathbf D \odot \mathbf H.
$$

**Обратный проход с Dropout.**  
При обратном распространении градиент умножается на ту же маску:
$$
\frac{\partial L}{\partial \mathbf H} = \frac{\partial L}{\partial \mathbf H'} \odot \mathbf D.
$$
Это гарантирует, что градиенты по отключённым нейронам равны нулю.

**Масштабирование на этапе тестирования.**  
На этапе тестирования dropout отключается, но чтобы сохранить ожидаемое значение выхода, необходимо умножить выходы на $(1-p)$. Действительно, математическое ожидание $h'$ равно $(1-p) h$. Поэтому на тесте используют $\mathbf h_{\text{test}} = (1-p) \mathbf h$.

**Inverted Dropout.**  
Чтобы избежать масштабирования на тесте, применяют обратный дропаут: во время обучения выходы умножаются на $\frac{1}{1-p}$:
$$
\mathbf h' = \frac{1}{1-p} \mathbf d \odot \mathbf h.
$$
Тогда на этапе тестирования масштабирование не требуется.

**Вероятностные характеристики.**  
Вероятность отключения всего слоя из $k$ нейронов равна $p^k$. Математическое ожидание числа активных нейронов $N$ равно $(1-p)k$, дисперсия — $k p (1-p)$.

**Alpha Dropout.**  
Для функции активации SELU, обладающей свойством самонормализации, обычный dropout нарушает нормализацию (дисперсия выхода становится $\frac{p}{1-p}\mathrm{Var}(h)$). Alpha Dropout заменяет отключённый выход не нулём, а константой $\alpha'$, и применяет аффинное преобразование $a \cdot x + b$, чтобы сохранить нулевое среднее и единичную дисперсию. Параметры $a$ и $b$ находятся из системы:
$$
E(a \cdot (d \cdot x + (1-d)\alpha') + b) = 0,
$$
$$
\mathrm{Var}(a \cdot (d \cdot x + (1-d)\alpha') + b) = 1.
$$
Решение:
$$
a = (p + (\alpha')^2 p(1-p))^{-1/2}, \quad b = -a (1-p) \alpha'.
$$

#### 4.4. Weight Decay и AdamW

В классическом градиентном спуске L2-регуляризация эквивалентна weight decay: обновление весов принимает вид $w := w - \eta (\nabla L + \lambda w) = (1-\eta\lambda)w - \eta \nabla L$. Однако в адаптивных оптимизаторах, таких как Adam, регуляризационный член взаимодействует с накопленными статистиками градиентов, что приводит к неравномерному затуханию весов и снижению эффективности регуляризации.

В Adam с L2-регуляризацией градиент $g_t$ заменяется на $g_t + \lambda w_{t-1}$, а затем применяются моменты и нормализация. Из-за деления на $\sqrt{\hat{v}_t} + \epsilon$ затухание весов становится зависимым от истории градиентов. Это может приводить к переобучению.

**AdamW** (Adam with decoupled weight decay) разделяет обновление от градиента и weight decay:
$$
w_t = (1 - \eta \lambda) w_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon},
$$
где $\hat{m}_t, \hat{v}_t$ вычисляются как в Adam, но градиент $g_t$ не содержит регуляризационного члена. Такой подход позволяет более точно контролировать регуляризацию и обычно даёт лучшие результаты.

---

### 5. Инициализация весов

Правильная инициализация начальных значений весов критически важна для успешного обучения глубоких сетей. Слишком малые или слишком большие начальные веса могут привести к затуханию или взрыву градиентов.

#### 5.1. Проблема случайной инициализации

При инициализации весов из нормального распределения $\mathcal N(0, \sigma^2)$ дисперсия выходов и градиентов может быстро изменяться при переходе от слоя к слою. Если $\sigma$ слишком велика, активации насыщаются (для сигмоиды/tanh) или растут неограниченно (для ReLU); если $\sigma$ мала — сигнал затухает.

Рассмотрим линейный слой с $n_{\text{in}}$ входами и весами $w_i$, инициализированными из $U[-\frac{1}{\sqrt{n_{\text{in}}}}, \frac{1}{\sqrt{n_{\text{in}}}}]$. Дисперсия веса равна $\frac{1}{3 n_{\text{in}}}$. Если входы имеют нулевое среднее и дисперсию $\mathrm{Var}(x)$, то дисперсия выхода одного нейрона (без активации) будет $n_{\text{in}} \cdot \mathrm{Var}(x) \cdot \frac{1}{3 n_{\text{in}}} = \frac{1}{3}\mathrm{Var}(x)$. То есть дисперсия уменьшается втрое на каждом слое, что приводит к быстрому затуханию сигнала и градиентов.

#### 5.2. Xavier/Glorot инициализация

Чтобы сохранить дисперсию сигнала при прямом и обратном распространении, Xavier Glorot и Yoshua Bengio предложили инициализировать веса из равномерного распределения
$$
w_{ij} \sim U\left[-\frac{\sqrt{6}}{\sqrt{n_{\text{in}} + n_{\text{out}}}},\; \frac{\sqrt{6}}{\sqrt{n_{\text{in}} + n_{\text{out}}}}\right],
$$
или из нормального с дисперсией
$$
\sigma^2 = \frac{2}{n_{\text{in}} + n_{\text{out}}}.
$$
Этот компромисс учитывает и прямое ($n_{\text{in}}$ слагаемых), и обратное ($n_{\text{out}}$ градиентов) распространение.

#### 5.3. He инициализация

Для ReLU-активации Kaiming He предложил использовать дисперсию
$$
\sigma^2 = \frac{2}{n_{\text{in}}},
$$
поскольку ReLU обнуляет примерно половину входов (для симметричного распределения входов), и для сохранения дисперсии необходимо удвоить её. На практике He инициализация хорошо работает с ReLU и её вариантами.

---

### 6. Продвинутые оптимизаторы

Обычный градиентный спуск с постоянной скоростью обучения $\eta$ может сходиться медленно или застревать в седловых точках. Разработаны улучшенные алгоритмы оптимизации.

#### 6.1. Momentum

Метод моментума накапливает экспоненциально затухающее среднее прошлых градиентов:
$$
\mathbf v_t = \gamma \mathbf v_{t-1} + \eta \nabla_\theta L(\theta_{t-1}), \qquad \theta_t = \theta_{t-1} - \mathbf v_t,
$$
где $\gamma \in [0,1)$ — коэффициент моментума (обычно 0.9). Это позволяет преодолевать небольшие локальные колебания и ускорять движение в направлениях с устойчивым градиентом.

#### 6.2. Nesterov Accelerated Gradient (NAG)

NAG — модификация моментума, при которой градиент вычисляется не в текущей, а в «предварительной» точке:
$$
\mathbf v_t = \gamma \mathbf v_{t-1} + \eta \nabla_\theta L(\theta_{t-1} - \gamma \mathbf v_{t-1}), \qquad \theta_t = \theta_{t-1} - \mathbf v_t.
$$
Это улучшает сходимость для выпуклых задач и часто даёт более точные шаги.

#### 6.3. AdaGrad

AdaGrad адаптирует скорость обучения для каждого параметра, накапливая сумму квадратов градиентов:
$$
\mathbf G_t = \mathbf G_{t-1} + \mathbf g_t \odot \mathbf g_t, \qquad
\theta_{t,i} = \theta_{t-1,i} - \frac{\eta}{\sqrt{G_{t,ii} + \epsilon}} g_{t,i},
$$
где $\mathbf g_t = \nabla_\theta L(\theta_{t-1})$, $\epsilon$ — малое число для численной устойчивости. Недостаток — монотонное накопление $G_{t,ii}$ приводит к слишком малой скорости обучения на поздних итерациях.

#### 6.4. RMSProp

RMSProp устраняет недостаток AdaGrad, используя экспоненциально затухающее среднее квадратов градиентов:
$$
\mathbf G_t = \beta \mathbf G_{t-1} + (1-\beta) \mathbf g_t \odot \mathbf g_t, \qquad
\theta_t = \theta_{t-1} - \frac{\eta}{\sqrt{\mathbf G_t + \epsilon}} \odot \mathbf g_t,
$$
где $\beta \approx 0.9$. Это позволяет сохранять разумную скорость обучения в течение всего процесса.

#### 6.5. Adam

Adam (Adaptive Moment Estimation) объединяет идеи Momentum и RMSProp. Поддерживаются две экспоненциально затухающие оценки: первая для среднего градиента $\mathbf m_t$, вторая для среднего квадрата градиента $\mathbf v_t$:
$$
\mathbf m_t = \beta_1 \mathbf m_{t-1} + (1-\beta_1)\mathbf g_t, \qquad
\mathbf v_t = \beta_2 \mathbf v_{t-1} + (1-\beta_2)\mathbf g_t^2.
$$
Поскольку начальные $\mathbf m_0=\mathbf 0$, $\mathbf v_0=\mathbf 0$ смещают оценки к нулю, применяется коррекция смещения:
$$
\hat{\mathbf m}_t = \frac{\mathbf m_t}{1-\beta_1^t}, \qquad
\hat{\mathbf v}_t = \frac{\mathbf v_t}{1-\beta_2^t}.
$$
Обновление параметров:
$$
\theta_t = \theta_{t-1} - \frac{\eta}{\sqrt{\hat{\mathbf v}_t} + \epsilon} \hat{\mathbf m}_t.
$$
Рекомендуемые значения: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$. Adam является одним из наиболее популярных оптимизаторов.

---

### 7. Проблема исчезающего и взрывающегося градиента

#### 7.1. Причины

При обратном распространении градиенты умножаются на производные функций активации и веса. Если производные меньше 1 (например, у сигмоиды максимум 0.25) и веса малы, произведение может экспоненциально убывать при переходе к начальным слоям — градиент **исчезает**. Если веса велики или используется ReLU без ограничения, произведение может экспоненциально расти — градиент **взрывается**. Обе проблемы затрудняют обучение глубоких сетей.

#### 7.2. Методы борьбы

- **Выбор активации**: ReLU и её варианты уменьшают исчезновение градиента для положительных входов.
- **Инициализация**: Xavier и He инициализации поддерживают дисперсию сигнала.
- **Нормализация по батчам (Batch Normalization)**: нормализация входов каждого слоя по мини-батчу стабилизирует распределение активаций и градиентов.
- **Обрезка градиента (gradient clipping)**: при превышении нормой градиента порога градиент масштабируется.
- **Residual connections**: введение «коротких замыканий» $\mathbf a^{(\ell)} = \sigma(\mathbf z^{(\ell)}) + \mathbf a^{(\ell-1)}$ облегчает протекание градиента.

#### 7.3. Batch Normalization

**Мотивация.** Внутренний ковариационный сдвиг — изменение распределения входов каждого слоя в процессе обучения — замедляет сходимость. BN нормализует активации по мини-батчу.

**Алгоритм на этапе обучения.**  
Для мини-батча $\mathcal B$ размера $m$ и каждой активации $z_i^{(\ell)}$ (до функции активации) вычисляются среднее и дисперсия:
$$
\mu_{\mathcal B} = \frac{1}{m} \sum_{k=1}^{m} z_k, \qquad
\sigma_{\mathcal B}^2 = \frac{1}{m} \sum_{k=1}^{m} (z_k - \mu_{\mathcal B})^2.
$$
Нормализация:
$$
\hat z_k = \frac{z_k - \mu_{\mathcal B}}{\sqrt{\sigma_{\mathcal B}^2 + \epsilon}}.
$$
Затем применяется аффинное преобразование с обучаемыми параметрами $\gamma, \beta$:
$$
y_k = \gamma \hat z_k + \beta.
$$
Во время обучения также обновляются скользящие средние:
$$
\mu_{\text{running}} \leftarrow \alpha \mu_{\text{running}} + (1-\alpha)\mu_{\mathcal B},
$$
$$
\sigma^2_{\text{running}} \leftarrow \alpha \sigma^2_{\text{running}} + (1-\alpha)\sigma^2_{\mathcal B},
$$
где $\alpha$ близко к 1 (например, 0.9).

**На этапе тестирования** используются $\mu_{\text{running}}$ и $\sigma^2_{\text{running}}$, а не статистики батча.

**Преимущества BN.** Ускорение сходимости, снижение чувствительности к инициализации, регуляризующий эффект.

**Замечание.** BN обычно применяется до функции активации, хотя возможны варианты.

---

### 8. Обучение по мини-батчам

В реальных задачах обучающая выборка содержит тысячи или миллионы примеров. Вычисление градиента по всей выборке на каждом шаге (full-batch GD) вычислительно дорого. Стохастический градиентный спуск (SGD) использует один случайный пример за шаг, что шумно, но эффективно. Компромиссом является мини-батч GD: градиент вычисляется по небольшому подмножеству (батчу) размера $m$.

Пусть $\mathbf X \in \mathbb R^{d \times m}$ — матрица $m$ входных векторов, $\mathbf Y$ — соответствующая матрица целевых значений. Прямой проход в матричной форме:
$$
\mathbf Z^{(\ell)} = \mathbf W^{(\ell)} \mathbf A^{(\ell-1)} + \mathbf b^{(\ell)} \mathbf 1_m^\top, \qquad
\mathbf A^{(\ell)} = \sigma(\mathbf Z^{(\ell)}).
$$
Функция потерь для батча усредняется:
$$
L = \frac{1}{m} \sum_{k=1}^{m} L_k,
$$
где $L_k$ — потеря на $k$-м примере. Градиенты по параметрам также усредняются:
$$
\frac{\partial L}{\partial \mathbf W^{(\ell)}} = \frac{1}{m} \sum_{k=1}^{m} \frac{\partial L_k}{\partial \mathbf W^{(\ell)}}.
$$
Матричная реализация позволяет эффективно использовать параллельные вычисления на GPU. Типичный размер мини-батча — от 32 до 256.

---

### 9. Многоклассовая классификация и вывод градиентов для softmax + cross-entropy

#### 9.1. Постановка задачи

Рассмотрим задачу классификации с $C$ классами. Выходной слой MLP содержит $C$ нейронов, а в качестве функции активации используется **softmax**, которая преобразует вектор логитов $\mathbf z = (z_1, \dots, z_C)^\top$ в вектор вероятностей $\mathbf s = (s_1, \dots, s_C)^\top$:
$$
s_i = \mathrm{softmax}(\mathbf z)_i = \frac{e^{z_i}}{\sum_{k=1}^{C} e^{z_k}}, \quad i = 1, \dots, C.
$$
Заметим, что $\sum_{i=1}^{C} s_i = 1$ и $s_i > 0$. В качестве целевого значения используется **one-hot** вектор $\mathbf y = (y_1, \dots, y_C)^\top$, где $y_i \in \{0,1\}$ и $\sum_i y_i = 1$. Функция потерь — **категориальная кросс-энтропия**:
$$
L = -\sum_{i=1}^{C} y_i \log s_i.
$$
Наша цель — вычислить градиент $L$ по логитам $\mathbf z$, то есть $\partial L / \partial z_j$ для каждого $j$.

#### 9.2. Вывод градиента по логитам

Применим цепное правило. Так как $L$ зависит от $\mathbf z$ через все $s_i$, имеем:
$$
\frac{\partial L}{\partial z_j} = \sum_{i=1}^{C} \frac{\partial L}{\partial s_i} \cdot \frac{\partial s_i}{\partial z_j}.
$$

**Шаг 1.** Найдём $\partial L / \partial s_i$. Поскольку $L = -\sum_{k=1}^{C} y_k \log s_k$, то
$$
\frac{\partial L}{\partial s_i} = -\frac{y_i}{s_i}.
$$

**Шаг 2.** Найдём $\partial s_i / \partial z_j$ — производную softmax. Здесь нужно рассмотреть два случая: когда $i = j$ и когда $i \neq j$.

Запишем $s_i = e^{z_i} / S$, где $S = \sum_{k=1}^{C} e^{z_k}$. Тогда:

- Если $i = j$:
  $$
  \frac{\partial s_i}{\partial z_i} = \frac{e^{z_i} \cdot S - e^{z_i} \cdot e^{z_i}}{S^2} = \frac{e^{z_i}}{S} \left(1 - \frac{e^{z_i}}{S}\right) = s_i (1 - s_i).
  $$

- Если $i \neq j$:
  $$
  \frac{\partial s_i}{\partial z_j} = -\frac{e^{z_i} \cdot e^{z_j}}{S^2} = -s_i s_j.
  $$

Объединяя оба случая, можно записать с использованием символа Кронекера $\delta_{ij}$:
$$
\frac{\partial s_i}{\partial z_j} = s_i (\delta_{ij} - s_j).
$$

**Шаг 3.** Подставим найденные производные в сумму:
$$
\frac{\partial L}{\partial z_j} = \sum_{i=1}^{C} \left(-\frac{y_i}{s_i}\right) \cdot s_i (\delta_{ij} - s_j)
= -\sum_{i=1}^{C} y_i \delta_{ij} + \sum_{i=1}^{C} y_i s_j.
$$

Первое слагаемое: $\sum_{i=1}^{C} y_i \delta_{ij} = y_j$, так как только при $i = j$ символ Кронекера отличен от нуля. Второе слагаемое: $s_j \sum_{i=1}^{C} y_i = s_j \cdot 1 = s_j$, поскольку $\mathbf y$ — one-hot вектор и его компоненты в сумме дают 1.

Таким образом,
$$
\frac{\partial L}{\partial z_j} = s_j - y_j.
$$
В векторной форме:
$$
\frac{\partial L}{\partial \mathbf z} = \mathbf s - \mathbf y.
$$

#### 9.3. Свойства полученного градиента

Полученное выражение удивительно простое и интуитивно понятное: градиент по логитам равен разности предсказанной вероятности и истинной one-hot метки. Это означает, что обучение корректирует логиты ровно на величину ошибки предсказания. Кроме того, градиент не содержит множителя $s_j(1-s_j)$, который присутствовал бы при использовании MSE с сигмоидой, что устраняет проблему насыщения и ускоряет сходимость.

#### 9.4. Матричная форма для мини-батча

Пусть $\mathbf Z \in \mathbb R^{C \times m}$ — матрица логитов для $m$ примеров батча, где столбец $k$ соответствует логитам $k$-го примера. После применения softmax по столбцам получаем матрицу вероятностей $\mathbf S \in \mathbb R^{C \times m}$. Матрица истинных меток $\mathbf Y \in \mathbb R^{C \times m}$ содержит one-hot векторы по столбцам.

Функция потерь для батча (средняя по примерам):
$$
L = -\frac{1}{m} \sum_{k=1}^{m} \sum_{i=1}^{C} Y_{ik} \log S_{ik}.
$$
Градиент по $\mathbf Z$ (матрица частных производных по каждому элементу) равен
$$
\frac{\partial L}{\partial \mathbf Z} = \frac{1}{m} (\mathbf S - \mathbf Y).
$$

#### 9.5. Пример вычисления для одного примера

Пусть $C=3$, логиты $\mathbf z = (2.0, 1.0, 0.1)^\top$, истинный класс — второй ($\mathbf y = (0, 1, 0)^\top$). Вычислим softmax:
$$
S = e^{2.0} + e^{1.0} + e^{0.1} \approx 7.389 + 2.718 + 1.105 = 11.212,
$$
$$
s_1 = 7.389 / 11.212 \approx 0.659, \quad
s_2 = 2.718 / 11.212 \approx 0.242, \quad
s_3 = 1.105 / 11.212 \approx 0.099.
$$
Функция потерь: $L = -\log s_2 \approx -\log(0.242) \approx 1.419$. Градиент:
$$
\frac{\partial L}{\partial \mathbf z} = \mathbf s - \mathbf y = (0.659, 0.242-1, 0.099)^\top = (0.659, -0.758, 0.099)^\top.
$$
Заметим, что компонента для истинного класса отрицательна, что при обновлении весов уменьшит соответствующий логит, а для остальных положительна, увеличивая их.

---

### 10. Практические аспекты

#### 10.1. Нормализация входных данных

Перед обучением входные признаки обычно масштабируются, например, к нулевому среднему и единичной дисперсии:
$$
x_i' = \frac{x_i - \mu_i}{\sigma_i}.
$$
Это ускоряет сходимость градиентного спуска и делает обучение более стабильным.

#### 10.2. Выбор гиперпараметров

- **Скорость обучения $\eta$** — ключевой гиперпараметр. Слишком малая замедляет обучение, слишком большая приводит к расходимости. Часто начинают с $\eta=0.001$ и корректируют.
- **Размер мини-батча** $m$ — компромисс между шумом и вычислительной эффективностью.
- **Число эпох** — полный проход по обучающей выборке. Используют раннюю остановку при ухудшении ошибки на валидационной выборке.
- **Коэффициент регуляризации** $\lambda$ — подбирается по валидационной ошибке.

#### 10.3. Критерии остановки

- Достижение заданного числа эпох.
- Отсутствие улучшения ошибки на валидационной выборке в течение нескольких эпох (early stopping).
- Достижение требуемой точности на обучающей выборке.

#### 10.4. Метрики качества

Для классификации используют accuracy, precision, recall, F1-score, AUC-ROC. Для регрессии — MSE, MAE, $R^2$. Выбор метрики зависит от задачи.

---

### 11. Теорема универсальной аппроксимации

#### 11.1. Историческая справка и формулировка

Теорема универсальной аппроксимации для нейронных сетей была доказана Джорджем Цыбенко в 1989 году. Она утверждает, что многослойный перцептрон с одним скрытым слоем, содержащим конечное число нейронов с сигмоидной функцией активации, способен аппроксимировать любую непрерывную функцию на компактном множестве с произвольной точностью.

**Теорема (Цыбенко, 1989).**  
Пусть $\sigma$ — непрерывная, непостоянная, ограниченная и монотонно возрастающая функция. Для любой непрерывной функции $f: K \to \mathbb R$, где $K \subset \mathbb R^n$ — компакт, и для любого $\varepsilon > 0$ существуют целое число $N$, векторы $\mathbf w_i \in \mathbb R^n$, числа $b_i, \alpha_i \in \mathbb R$ такие, что
$$
\left| f(\mathbf x) - \sum_{i=1}^{N} \alpha_i \, \sigma(\mathbf w_i^\top \mathbf x + b_i) \right| < \varepsilon
$$
для всех $\mathbf x \in K$.

Другими словами, множество функций вида $\sum_{i=1}^{N} \alpha_i \sigma(\mathbf w_i^\top \mathbf x + b_i)$ плотно в пространстве непрерывных функций на компакте $K$ относительно равномерной нормы.

#### 11.2. Интуитивное объяснение

Каждый нейрон скрытого слоя $\sigma(\mathbf w_i^\top \mathbf x + b_i)$ можно рассматривать как «ступеньку» в многомерном пространстве. Меняя веса $\mathbf w_i$, мы вращаем направление ступеньки; меняя смещение $b_i$, сдвигаем её; меняя выходной вес $\alpha_i$, управляем высотой. Комбинируя множество таких ступенек, можно приблизить произвольную непрерывную функцию, подобно тому как кусочно-постоянные функции приближают гладкие.

#### 11.3. Эскиз доказательства

Доказательство теоремы обычно опирается на следующие факты.

1. **Аппроксимация ступенчатыми функциями.** Любую непрерывную функцию на компакте можно равномерно приблизить простыми функциями, например, кусочно-постоянными.

2. **Аппроксимация ступенек сигмоидами.** Сигмоидная функция $\sigma(z)$ при больших по модулю значениях $z$ ведёт себя как ступенька: $\sigma(z) \to 1$ при $z \to +\infty$ и $\sigma(z) \to 0$ при $z \to -\infty$. Масштабируя и сдвигая аргумент, можно получить узкие «переходные» области, имитирующие индикаторы множеств.

3. **Теорема Стоуна–Вейерштрасса.** Множество конечных линейных комбинаций сигмоид удовлетворяет условиям этой теоремы (замкнуто относительно сложения, умножения и разделяет точки), поэтому его замыкание совпадает с пространством непрерывных функций.

Более строгое доказательство можно найти в оригинальной статье Цыбенко или в учебниках по теории нейронных сетей.

#### 11.4. Обобщения

Теорема была обобщена на другие функции активации. Например, в 1991 году Лешковец и соавторы показали, что достаточно непостоянной ограниченной непрерывной функции активации. Позднее было доказано, что ReLU-сети также обладают универсальной аппроксимационной способностью: любая непрерывная функция на компакте может быть сколь угодно точно приближена сетью с одним скрытым слоем и ReLU-активацией (при достаточно большом числе нейронов).

#### 11.5. Ограничения теоремы

Несмотря на фундаментальную важность, теорема универсальной аппроксимации имеет ряд ограничений.

- **Число нейронов может быть огромным.** Теорема не даёт оценок на $N$ в зависимости от $\varepsilon$ и сложности функции. На практике требуемое число нейронов может быть экспоненциально большим.
- **Не гарантирует обучаемость.** Теорема утверждает существование набора параметров, но не гарантирует, что градиентный спуск найдёт его. Оптимизация может застрять в локальных минимумах или седловых точках.
- **Глубина имеет значение.** Хотя один скрытый слой теоретически достаточен, глубокие сети часто могут представлять те же функции с гораздо меньшим числом нейронов и лучше обобщать.
- **Непрерывность и компактность.** Теорема применима только к непрерывным функциям на компактах; для разрывных функций или неограниченных областей она не действует.
- **Обобщение.** Теорема не учитывает конечность обучающей выборки и проблему переобучения.

Таким образом, теорема универсальной аппроксимации является важным теоретическим обоснованием выразительной мощности MLP, но практические аспекты обучения и выбора архитектуры требуют дополнительных соображений.

---

### 12. Заключение второй части

Во второй части были рассмотрены ключевые практические и теоретические аспекты обучения многослойных перцептронов: разнообразие функций активации и потерь, методы регуляризации и инициализации, современные оптимизаторы, проблемы градиентов и их решения, обучение по мини-батчам, многоклассовая классификация, практические рекомендации и теорема универсальной аппроксимации. Эти сведения дополняют базовый материал первой части и дают достаточно полное представление о современных MLP.